In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import shutil
import cv2
from tifffile import imread, imwrite
from towbintools.foundation.file_handling import get_dir_filemap, add_dir_to_experiment_filemap
from skimage.measure import label, regionprops
from scipy.ndimage import binary_fill_holes
from towbintools.foundation.image_handling import read_tiff_file

# Annotated Crops

In [2]:
db_dir = "/mnt/towbin.data/shared/spsalmon/stardist_database/443_60x_epidermal_classification/db2_part1"

img_dir = os.path.join(db_dir, "raw")
annot_dir = os.path.join(db_dir, "mask")

processed_img_dir = os.path.join(db_dir, "processed_imgs")
processed_annot_dir = os.path.join(db_dir, "processed_annotations")

os.makedirs(processed_img_dir, exist_ok=True)
os.makedirs(processed_annot_dir, exist_ok=True)

img_paths = sorted([os.path.join(img_dir, f) for f in os.listdir(img_dir)])
annot_paths = sorted([os.path.join(annot_dir, f) for f in os.listdir(annot_dir)])
# replace .tif with .tiff in annot_paths
new_annot_paths = [p.replace(".tif", ".tiff") for p in annot_paths]

img_paths = [p for p in img_paths if os.path.basename(p) in [os.path.basename(a) for a in new_annot_paths]]
img_paths = sorted(img_paths)
annot_paths = sorted(annot_paths)

print(f"Found {len(img_paths)} images and {len(annot_paths)} annotations.")

Found 81 images and 81 annotations.


In [3]:
def process_crop_annotations(annotations):
    labels = label(annotations, connectivity=2)
    processed_annotations = np.zeros_like(labels)
    for lbl in np.unique(labels):
        if lbl == 0:
            continue
        mask = labels == lbl
        mask = binary_fill_holes(mask)
        processed_annotations[mask] = lbl
    return processed_annotations

for img_path, annot_path in zip(img_paths, annot_paths):
    print(f"Processing {os.path.basename(img_path)} and {os.path.basename(annot_path)}")
    img = read_tiff_file(img_path)
    annot = read_tiff_file(annot_path)

    processed_annot = process_crop_annotations(annot)

    imwrite(os.path.join(processed_img_dir, os.path.basename(img_path)), img.astype(np.uint16), compression="zlib")
    imwrite(os.path.join(processed_annot_dir, os.path.basename(annot_path)), processed_annot.astype(np.uint16), compression="zlib")

Processing crop_00614761.tiff and crop_00614761.tif
Processing crop_023f0e40.tiff and crop_023f0e40.tif
Processing crop_026ee236.tiff and crop_026ee236.tif
Processing crop_047bbb4d.tiff and crop_047bbb4d.tif
Processing crop_049c8a48.tiff and crop_049c8a48.tif
Processing crop_04b240e5.tiff and crop_04b240e5.tif
Processing crop_05f726a6.tiff and crop_05f726a6.tif
Processing crop_0809813c.tiff and crop_0809813c.tif
Processing crop_0b5226d6.tiff and crop_0b5226d6.tif
Processing crop_0c5d7c72.tiff and crop_0c5d7c72.tif
Processing crop_0d209ea6.tiff and crop_0d209ea6.tif
Processing crop_0ddc76fc.tiff and crop_0ddc76fc.tif
Processing crop_102f77fa.tiff and crop_102f77fa.tif
Processing crop_14a03ad7.tiff and crop_14a03ad7.tif
Processing crop_1634f7d3.tiff and crop_1634f7d3.tif
Processing crop_17e85830.tiff and crop_17e85830.tif
Processing crop_195c9a4f.tiff and crop_195c9a4f.tif
Processing crop_23952ac5.tiff and crop_23952ac5.tif
Processing crop_24506f49.tiff and crop_24506f49.tif
Processing c

# Combine DBs

In [4]:
database_dir = "/mnt/towbin.data/shared/spsalmon/stardist_database/443_60x_epidermal_classification/"
# get all subdirectories in database_dir

output_dir = os.path.join(database_dir, "merged_database")

subdirs = [os.path.join(database_dir, d) for d in os.listdir(database_dir) if os.path.isdir(os.path.join(database_dir, d)) and d != "merged_database"]

os.makedirs(output_dir, exist_ok=True)

directories_to_merge = ["processed_imgs", "processed_annotations"]

for subdir in subdirs:
    for directory in directories_to_merge:
        if os.path.exists(os.path.join(output_dir, directory)):
            print(f"Merging {directory} from {subdir} into {output_dir}")
            for item in os.listdir(os.path.join(subdir, directory)):
                s = os.path.join(subdir, directory, item)
                d = os.path.join(output_dir, directory, item)
                if os.path.isdir(s):
                    shutil.copytree(s, d, dirs_exist_ok=True)
                else:
                    shutil.copy2(s, d)
                
        else:
            shutil.copytree(os.path.join(subdir, directory), os.path.join(output_dir, directory))


Merging processed_imgs from /mnt/towbin.data/shared/spsalmon/stardist_database/443_60x_epidermal_classification/db2_part1 into /mnt/towbin.data/shared/spsalmon/stardist_database/443_60x_epidermal_classification/merged_database
Merging processed_annotations from /mnt/towbin.data/shared/spsalmon/stardist_database/443_60x_epidermal_classification/db2_part1 into /mnt/towbin.data/shared/spsalmon/stardist_database/443_60x_epidermal_classification/merged_database
Merging processed_imgs from /mnt/towbin.data/shared/spsalmon/stardist_database/443_60x_epidermal_classification/initial_db into /mnt/towbin.data/shared/spsalmon/stardist_database/443_60x_epidermal_classification/merged_database
Merging processed_annotations from /mnt/towbin.data/shared/spsalmon/stardist_database/443_60x_epidermal_classification/initial_db into /mnt/towbin.data/shared/spsalmon/stardist_database/443_60x_epidermal_classification/merged_database


# Annotated Stacks

In [7]:
# def process_stack_annotations(annotations):

#     for i, plane in enumerate(annotations):
#     #     labels = label(annotations)

#     #     # for each label, fill the holes
#     #     for lbl in np.unique(labels):
#     #         if label == 0:
#     #             continue
#     #         mask = labels == lbl
#     #         mask = binary_fill_holes(mask)
#     #         labels[mask] = lbl
#         plane = binary_fill_holes(plane)
#         labels = label(plane)
#         annotations[i] = labels
#     return annotations
    

# for i, row in filemap.iterrows():
#     img_path = row["ImagePath"]
#     annot_path = row["Annotations"]
    
#     annotations = imread(annot_path)

#     annotations = (annotations > 0).astype(np.uint8)
#     img = read_tiff_file(img_path)

#     # get the idx of non zero planes
#     non_zero_planes = np.any(annotations, axis=(1, 2))
#     non_zero_planes = np.where(non_zero_planes)[0]

#     print(img.shape)

#     print(non_zero_planes)

#     annotations = annotations[non_zero_planes]
#     img = img[non_zero_planes]

#     annotations = process_annotations(annotations)

    
#     for (img_plane, annot_plane, plane) in zip(img, annotations, non_zero_planes):
#         img_plane_path = os.path.join(processed_img_dir, f"{plane}_{os.path.basename(img_path)}")
#         annot_plane_path = os.path.join(processed_annot_dir, f"{plane}_{os.path.basename(annot_path)}")
        
#         imwrite(img_plane_path, img_plane)
#         imwrite(annot_plane_path, annot_plane)
        
#         print(f"Saved {img_plane_path}")
#         print(f"Saved {annot_plane_path}")